In [16]:
#Import the libraries
import pandas as pd 

In [17]:
df =pd.read_csv('../data/raw/csic/csic_database.csv')

In [18]:
print("Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)

Shape: (61065, 17)

Column Names:
['Unnamed: 0', 'Method', 'User-Agent', 'Pragma', 'Cache-Control', 'Accept', 'Accept-encoding', 'Accept-charset', 'language', 'host', 'cookie', 'content-type', 'connection', 'lenght', 'content', 'classification', 'URL']

Data Types:
Unnamed: 0         object
Method             object
User-Agent         object
Pragma             object
Cache-Control      object
Accept             object
Accept-encoding    object
Accept-charset     object
language           object
host               object
cookie             object
content-type       object
connection         object
lenght             object
content            object
classification      int64
URL                object
dtype: object


In [19]:
#Checking the label distribution
print("Label Distribution:")
print(df['classification'].value_counts())
print(f"\nNormal:    {(df['classification']==0).sum():,}")
print(f"Anomalous: {(df['classification']==1).sum():,}")

Label Distribution:
classification
0    36000
1    25065
Name: count, dtype: int64

Normal:    36,000
Anomalous: 25,065


In [20]:
#Checking for missing values 
print("\nMissing Values")
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_%': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing[missing['missing_count'] > 0])


Missing Values
              missing_count  missing_%
Accept                  397       0.65
content-type          43088      70.56
lenght                43088      70.56
content               43088      70.56


In [21]:
#Checking for duplicated rows
print(f"\nDuplicate rows: {df.duplicated().sum():,}")

#Sample of URL column
print("\n Sample URLs ")
print(df['URL'].dropna().head(5).tolist())

#Sample of content column
print("\nSample Content/Body")
print(df['content'].dropna().head(5).tolist())

#Unique HTTP methods
print("\nHTTP Methods")
print(df['Method'].value_counts())



Duplicate rows: 0

 Sample URLs 
['http://localhost:8080/tienda1/index.jsp HTTP/1.1', 'http://localhost:8080/tienda1/publico/anadir.jsp?id=3&nombre=Vino+Rioja&precio=100&cantidad=55&B1=A%F1adir+al+carrito HTTP/1.1', 'http://localhost:8080/tienda1/publico/anadir.jsp HTTP/1.1', 'http://localhost:8080/tienda1/publico/autenticar.jsp?modo=entrar&login=choong&pwd=d1se3ci%F3n&remember=off&B1=Entrar HTTP/1.1', 'http://localhost:8080/tienda1/publico/autenticar.jsp HTTP/1.1']

Sample Content/Body
['id=3&nombre=Vino+Rioja&precio=100&cantidad=55&B1=A%F1adir+al+carrito', 'modo=entrar&login=choong&pwd=d1se3ci%F3n&remember=off&B1=Entrar', 'id=2', 'errorMsg=Credenciales+incorrectas', 'modo=insertar&precio=2672&B1=Pasar+por+caja']

HTTP Methods
Method
GET     43088
POST    17580
PUT       397
Name: count, dtype: int64


In [22]:
#Cleainng the data now 
import numpy as np

#Create a copy 
df_clean = df.copy()

#Drop Index column
df_clean = df_clean.drop(columns=['Unnamed: 0'])
print("Dropped Index column")

#Fixing the column name
df_clean = df_clean.rename(columns={
    'lenght':         'content_length',
    'classification': 'label',
    'content':        'body',
    'User-Agent':     'user_agent',
    'Cache-Control':  'cache_control',
    'Accept-encoding':'accept_encoding',
    'Accept-charset': 'accept_charset',
    'content-type':   'content_type',
    'connection':     'connection',
    'host':           'host',
    'cookie':         'cookie',
    'language':       'language',
    'Pragma':         'pragma',
    'Accept':         'accept',
    'Method':         'method',
})
print(" Fixed column names")
print("   New columns:", df_clean.columns.tolist())

#Clean URL Column
# Remove ' HTTP/1.1' from end of URL
df_clean['url_raw'] = df_clean['URL'].str.replace(
    r'\s+HTTP/\d\.\d$', '', regex=True
).str.strip()

# Extract jthe path (remove http://localhost:8080)
df_clean['url_path'] = df_clean['url_raw'].str.replace(
    r'https?://[^/]+', '', regex=True
)

# Extract query string
df_clean['query_string'] = df_clean['url_raw'].apply(
    lambda x: x.split('?')[1] if isinstance(x, str) and '?' in x else ''
)

# Drop the original messy URL column
df_clean = df_clean.drop(columns=['URL'])
print(" Cleaned URL column → url_path + query_string")

#Fill Missing values
# GET requests legitimately have no body — fill with empty string
df_clean['body']           = df_clean['body'].fillna('')
df_clean['content_type']   = df_clean['content_type'].fillna('none')
df_clean['content_length'] = df_clean['content_length'].fillna('0')
df_clean['accept']         = df_clean['accept'].fillna('unknown')
df_clean['cookie']         = df_clean['cookie'].fillna('none')
print(" Filled missing values")

# Standardise method to uppercase ───────────────────
df_clean['method'] = df_clean['method'].str.upper().str.strip()
print(" Standardised HTTP methods")

# Verify no unexpected nulls remain 
remaining_nulls = df_clean.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
if len(remaining_nulls) == 0:
    print("No nulls remaining")
else:
    print("Remaining nulls:")
    print(remaining_nulls)

# Final shape check 
print(f"\nClean dataset shape: {df_clean.shape}")
print(f" Columns: {df_clean.columns.tolist()}")
df_clean.head(3)


Dropped Index column
 Fixed column names
   New columns: ['method', 'user_agent', 'pragma', 'cache_control', 'accept', 'accept_encoding', 'accept_charset', 'language', 'host', 'cookie', 'content_type', 'connection', 'content_length', 'body', 'label', 'URL']
 Cleaned URL column → url_path + query_string
 Filled missing values
 Standardised HTTP methods
No nulls remaining

Clean dataset shape: (61065, 18)
 Columns: ['method', 'user_agent', 'pragma', 'cache_control', 'accept', 'accept_encoding', 'accept_charset', 'language', 'host', 'cookie', 'content_type', 'connection', 'content_length', 'body', 'label', 'url_raw', 'url_path', 'query_string']


,method,user_agent,pragma,cache_control,accept,accept_encoding,accept_charset,language,host,cookie,content_type,connection,content_length,body,label,url_raw,url_path,query_string
0,GET,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,JSESSIONID=1F767F17239C9B670A39E9B10C3825F4,none,close,0,,0,http://localhost:8080/tienda1/index.jsp,/tienda1/index.jsp,
1,GET,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5,none,close,0,,0,http://localhost:8080/tienda1/publico/anadir.j...,/tienda1/publico/anadir.jsp?id=3&nombre=Vino+R...,id=3&nombre=Vino+Rioja&precio=100&cantidad=55&...
2,POST,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,JSESSIONID=933185092E0B668B90676E0A2B0767AF,application/x-www-form-urlencoded,Connection: close,Content-Length: 68,id=3&nombre=Vino+Rioja&precio=100&cantidad=55&...,0,http://localhost:8080/tienda1/publico/anadir.jsp,/tienda1/publico/anadir.jsp,


In [23]:
#Checking cells for variance
print("=== Unique value counts per column ===\n")
for col in df_clean.columns:
    n = df_clean[col].nunique()
    print(f"{col:20} → {n:6} unique values")

=== Unique value counts per column ===

method               →      3 unique values
user_agent           →      1 unique values
pragma               →      1 unique values
cache_control        →      1 unique values
accept               →      2 unique values
accept_encoding      →      1 unique values
accept_charset       →      1 unique values
language             →      1 unique values
host                 →      2 unique values
cookie               →  61065 unique values
content_type         →      2 unique values
connection           →      2 unique values
content_length       →    383 unique values
body                 →  12092 unique values
label                →      2 unique values
url_raw              →  13498 unique values
url_path             →  13488 unique values
query_string         →  11857 unique values


In [24]:
print("Current columns in df_clean:")
print(df_clean.columns.tolist())


Current columns in df_clean:
['method', 'user_agent', 'pragma', 'cache_control', 'accept', 'accept_encoding', 'accept_charset', 'language', 'host', 'cookie', 'content_type', 'connection', 'content_length', 'body', 'label', 'url_raw', 'url_path', 'query_string']


In [25]:
# Dropping zero variance columns 
zero_variance_cols = [
    'user_agent',
    'pragma', 
    'cache_control',
    'accept_encoding',
    'accept_charset',
    'language',
    'url_raw'       # redundant — already have url_path
]

df_clean = df_clean.drop(columns=zero_variance_cols)

print(f"Dropped {len(zero_variance_cols)} useless columns")
print(f"Shape after dropping: {df_clean.shape}")
print(f"\nRemaining columns:")
for i, col in enumerate(df_clean.columns, 1):
    print(f"  {i:2}. {col}")

Dropped 7 useless columns
Shape after dropping: (61065, 11)

Remaining columns:
   1. method
   2. accept
   3. host
   4. cookie
   5. content_type
   6. connection
   7. content_length
   8. body
   9. label
  10. url_path
  11. query_string


In [26]:
df_clean.to_csv('../data/cleaned/csic_cleaned.csv', index=False)
print(f"Saved → data/cleaned/csic_cleaned.csv")
print(f"   Shape: {df_clean.shape}")

Saved → data/cleaned/csic_cleaned.csv
   Shape: (61065, 11)


In [27]:
print("Current columns:")
print(df_clean.columns.tolist())
print(f"\nShape: {df_clean.shape}")

Current columns:
['method', 'accept', 'host', 'cookie', 'content_type', 'connection', 'content_length', 'body', 'label', 'url_path', 'query_string']

Shape: (61065, 11)


In [28]:
# Check host column 
print("Host unique values:", df_clean['host'].unique())

#  Fix content_length — convert to numeric
df_clean['content_length'] = pd.to_numeric(
    df_clean['content_length'], errors='coerce'
).fillna(0).astype(int)

print("\ncontent_length converted to integer")
print("   Sample values:", df_clean['content_length'].value_counts().head())

# Validate label column
print(f"\nLabel values: {df_clean['label'].unique()}")
print(f"Label distribution:\n{df_clean['label'].value_counts()}")

# Check for any remaining nulls 
print("\n--- Remaining Nulls ---")
nulls = df_clean.isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls) == 0:
    print("Zero nulls — data is clean")
else:
    print(nulls)

# Final data types check 
print("\n--- Final Data Types ---")
print(df_clean.dtypes)

# ── 6. Final shape ────────────────────────────────────────
print(f"\nFinal clean shape: {df_clean.shape}")

Host unique values: ['localhost:8080' 'localhost:9090']

content_length converted to integer
   Sample values: content_length
0    61065
Name: count, dtype: int64

Label values: [0 1]
Label distribution:
label
0    36000
1    25065
Name: count, dtype: int64

--- Remaining Nulls ---
Zero nulls — data is clean

--- Final Data Types ---
method            object
accept            object
host              object
cookie            object
content_type      object
connection        object
content_length     int64
body              object
label              int64
url_path          object
query_string      object
dtype: object

Final clean shape: (61065, 11)


In [29]:
# Drop host — only lab addresses, zero real value 
df_clean = df_clean.drop(columns=['host'])
print("Dropped host column")

# Fix content_length properly 
# The issue is it converted everything to 0
# Let's check what the raw values actually look like
print("\nSample raw content_length values:")
print(df_clean['content_length'].value_counts().head(10))

# Recalculate content_length from actual body 
# More reliable than the original column
df_clean['content_length'] = df_clean['body'].apply(
    lambda x: len(str(x)) if x != '' else 0
)

print("\nRecalculated content_length from body column")
print("Sample values after fix:")
print(df_clean['content_length'].value_counts().head(10))
print(f"\nMax body length:  {df_clean['content_length'].max()}")
print(f"Mean body length: {df_clean['content_length'].mean():.2f}")
print(f"Zero length rows: {(df_clean['content_length']==0).sum():,}")

# ── 4. Final shape ────────────────────────────────────────
print(f"\n Shape: {df_clean.shape}")
print(f"Columns: {df_clean.columns.tolist()}")

Dropped host column

Sample raw content_length values:
content_length
0    61065
Name: count, dtype: int64

Recalculated content_length from body column
Sample values after fix:
content_length
0     43088
4      1057
33     1054
17     1046
38      529
43      527
34      460
5       458
18      452
74      360
Name: count, dtype: int64

Max body length:  836
Mean body length: 31.94
Zero length rows: 43,088

 Shape: (61065, 10)
Columns: ['method', 'accept', 'cookie', 'content_type', 'connection', 'content_length', 'body', 'label', 'url_path', 'query_string']


In [30]:
# Save final cleaned data
df_clean.to_csv('../data/cleaned/csic_cleaned_final.csv', index=False)
print(f"Saved → data/cleaned/csic_cleaned_final.csv")
print(f"   Shape: {df_clean.shape}")
print(f"   Rows: {len(df_clean):,}")
print(f"   Columns: {df_clean.shape[1]}")
print(f"\n DATA CLEANING COMPLETE")
print(f"   Normal traffic:    {(df_clean['label']==0).sum():,}")
print(f"   Anomalous traffic: {(df_clean['label']==1).sum():,}")

Saved → data/cleaned/csic_cleaned_final.csv
   Shape: (61065, 10)
   Rows: 61,065
   Columns: 10

 DATA CLEANING COMPLETE
   Normal traffic:    36,000
   Anomalous traffic: 25,065
